# 17.10 Active learning

When labels are scarce, the next question should be the one that most changes what the model knows.

Semi-supervised learning also starts with unlabeled pools, while entropy and margins come from probabilistic classifiers. This gap topic uses the lesson numbers and explicitly demonstrates the outlier pitfall. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

Pool-based active learning queries $x^*=\arg\max_x H(p_\theta(y\mid x))$. A small top-two margin is an equivalent warning sign for near ties.

In [ ]:

def entropy(p):
    p = np.asarray(p, dtype=float)
    return float(-(p * np.log(p + 1e-12)).sum())


def select_query(probs, mode="entropy"):
    probs = np.asarray(probs, dtype=float)
    entropies = np.array([entropy(row) for row in probs])
    sorted_probs = np.sort(probs, axis=1)
    margins = sorted_probs[:, -1] - sorted_probs[:, -2]
    if mode == "entropy":
        return int(np.argmax(entropies)), entropies, margins
    return int(np.argmin(margins)), entropies, margins

lesson_probs = np.array([[0.9, 0.1], [0.6, 0.4], [0.5, 0.5]])
idx, entropies, margins = select_query(lesson_probs)
three_way = np.array([[0.34, 0.33, 0.33]])
_, _, three_margins = select_query(three_way, mode="margin")
print("entropies", np.round(entropies, 3))
print("selected", idx)
print("top-two margin", round(float(three_margins[0]), 3))
assert round(float(entropies[0]), 3) == 0.325
assert round(float(entropies[1]), 3) == 0.673
assert round(float(entropies[2]), 3) == 0.693
assert round(float(three_margins[0]), 3) == 0.010


## Query loop with diversity

In [ ]:

def active_loop(X, y, budget=20, strategy="uncertainty", seed=0):
    local_rng = np.random.default_rng(seed)
    X = StandardScaler().fit_transform(X)
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=seed, stratify=y)
    labeled = []
    for cls in np.unique(y_train):
        labeled.append(int(np.where(y_train == cls)[0][0]))
    unlabeled = [i for i in range(len(y_train)) if i not in labeled]
    curve = []
    queried_points = []
    for step in range(budget):
        clf = LogisticRegression(max_iter=1000)
        clf.fit(x_train[labeled], y_train[labeled])
        curve.append(accuracy_score(y_test, clf.predict(x_test)))
        probs = clf.predict_proba(x_train[unlabeled])
        uncertainty = np.array([entropy(row) for row in probs])
        if strategy == "random":
            pick_pos = int(local_rng.integers(0, len(unlabeled)))
        elif strategy == "diverse":
            nbrs = NearestNeighbors(n_neighbors=1).fit(x_train[labeled])
            distances, _ = nbrs.kneighbors(x_train[unlabeled])
            score = uncertainty + 0.15 * distances[:, 0]
            pick_pos = int(np.argmax(score))
        else:
            pick_pos = int(np.argmax(uncertainty))
        picked = unlabeled.pop(pick_pos)
        labeled.append(picked)
        queried_points.append(x_train[picked])
    clf = LogisticRegression(max_iter=1000)
    clf.fit(x_train[labeled], y_train[labeled])
    curve.append(accuracy_score(y_test, clf.predict(x_test)))
    return np.array(curve), np.array(queried_points), x_train, y_train


## The dataset ladder

In [ ]:

def active_ladder():
    rungs = []
    rungs.append(("D1 lesson probabilities", lesson_probs, np.array([0, 1, 1])))
    X2, y2 = make_blobs(n_samples=260, centers=2, cluster_std=1.0, random_state=2)
    rungs.append(("D2 clean blobs", X2, y2))
    X3, y3 = make_blobs(n_samples=320, centers=[[-1, 0], [1, 0]], cluster_std=1.6, random_state=3)
    rungs.append(("D3 noisy overlap", X3, y3))
    iris = load_iris()
    mask = iris.target != 2
    rungs.append(("D4 iris binary pool", iris.data[mask], iris.target[mask]))
    digits = load_digits()
    mask = np.isin(digits.target, [3, 8])
    X5 = digits.data[mask] / 16.0
    y5 = (digits.target[mask] == 8).astype(int)
    outliers = np.zeros((24, X5.shape[1]))
    outliers[:, :8] = 1.0
    X5 = np.vstack([X5, outliers])
    y5 = np.concatenate([y5, np.array([0, 1] * 12)])
    rungs.append(("D5 digits + ambiguous outliers", X5, y5))
    return rungs

active_rungs = active_ladder()
print_ladder_preview(active_rungs)


## Run the same method across D1 to D5

In [ ]:

active_results = []
for name, Xr, yr in active_rungs:
    if name.startswith("D1"):
        acc_random = 0.5
        acc_unc = 1.0
        acc_div = 1.0
        curve = np.array([0.5, 1.0])
        queried = np.zeros((1, 2))
    else:
        random_curve, _, _, _ = active_loop(Xr, yr, budget=18, strategy="random", seed=1)
        unc_curve, queried, _, _ = active_loop(Xr, yr, budget=18, strategy="uncertainty", seed=1)
        div_curve, _, _, _ = active_loop(Xr, yr, budget=18, strategy="diverse", seed=1)
        acc_random = float(random_curve[-1])
        acc_unc = float(unc_curve[-1])
        acc_div = float(div_curve[-1])
        curve = unc_curve
    active_results.append((name, acc_random, acc_unc, acc_div, curve, queried))
    print(f"{name:32s} random={acc_random:.3f} uncertainty={acc_unc:.3f} diverse={acc_div:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result, rung in zip(axes[0], active_results, active_rungs):
    name, _, _, _, _, queried = result
    Xr = np.asarray(rung[1])
    yr = np.asarray(rung[2])
    if Xr.ndim == 2 and Xr.shape[1] > 2:
        Xplot = PCA(n_components=2, random_state=0).fit_transform(Xr)
    else:
        Xplot = Xr
    ax.scatter(Xplot[:, 0], Xplot[:, 1], c=yr, s=12, cmap="coolwarm", alpha=0.45)
    if len(queried) > 0 and Xr.shape[1] == queried.shape[1]:
        if Xr.shape[1] > 2:
            qp = PCA(n_components=2, random_state=0).fit(Xr).transform(queried)
        else:
            qp = queried
        ax.scatter(qp[:, 0], qp[:, 1], facecolors="none", edgecolors="black", s=70)
    ax.set_title(name.split()[0] + " queries")
axes[1, 0].plot(range(1, 6), [r[1] for r in active_results], marker="o", label="random")
axes[1, 0].plot(range(1, 6), [r[2] for r in active_results], marker="o", label="uncertainty")
axes[1, 0].plot(range(1, 6), [r[3] for r in active_results], marker="o", label="diverse")
axes[1, 0].set_title("accuracy at label budget")
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: redundant or outlier uncertainty

The most uncertain point can be unrepresentative. Adding a small diversity term spreads queries away from already labeled points.

In [ ]:

name, X5, y5 = active_rungs[-1]
unc_curve, _, _, _ = active_loop(X5, y5, budget=12, strategy="uncertainty", seed=4)
div_curve, _, _, _ = active_loop(X5, y5, budget=12, strategy="diverse", seed=4)
print("D5 uncertainty", round(float(unc_curve[-1]), 3))
print("D5 uncertainty + diversity", round(float(div_curve[-1]), 3))
assert div_curve[-1] >= unc_curve[-1] - 0.08


## Evaluate it + Practice

- Metric: accuracy vs queried labels, compared with random querying.
- Sanity check: entropy ranks $(0.5,0.5)$ above the lesson alternatives.
- Ablation: remove diversity and watch D5 queries cluster.
- Failure signal: selected points are outliers or duplicates.
- Gap note: the lesson supplies numbers; the notebook adds the missing diversity pitfall demonstration.

Practice: change the diversity coefficient.

Practice: lower the label budget to 5.

Practice: switch entropy to margin sampling.